# Portfolio Optimization with PCE (Pauli Correlation Encoding)

This notebook demonstrates portfolio optimization using **Pauli Correlation Encoding (PCE)** — a technique that compresses N binary decision variables into far fewer qubits:

| Encoding | Qubits for N variables | Example (N=20) |
|----------|----------------------|----------------|
| QAOA | N | 20 qubits |
| PCE Dense | O(log₂ N) | ~5 qubits |
| PCE Poly | O(√N) | ~5 qubits |

**Why PCE?** For large portfolios, QAOA requires one qubit per asset. PCE encodes each binary variable as the parity of a qubit subset, enabling dramatically fewer qubits while still solving the same QUBO.

**Workflow:**
1. Load financial data and partition assets (identical to the QAOA notebook)
2. Build QUBO matrices for each partition
3. Solve each partition using **PCE** instead of QAOA
4. Compare PCE solutions against the classical exact solver

## 1. Setup & Backend Selection

In [ ]:
import numpy as np
import math

from divi.qprog import PCE
from divi.qprog.optimizers import PymooOptimizer, PymooMethod
from divi.backends import ParallelSimulator, QoroService, JobConfig

### Choose Your Backend

PCE's qubit compression makes it especially attractive for **cloud execution** — even large portfolios need only a handful of qubits. Set `USE_CLOUD = True` to run on QoroService.

Get your API key at [dash.qoroquantum.net](https://dash.qoroquantum.net).

In [ ]:
USE_CLOUD = False  # Set to True to use QoroService cloud backend
SHOTS = 10_000

if USE_CLOUD:
    # QoroService automatically reads QORO_API_KEY from your environment
    backend = QoroService(config=JobConfig(shots=SHOTS))
    print("☁️  Using QoroService cloud backend")
else:
    backend = ParallelSimulator(shots=SHOTS)
    print("💻 Using local ParallelSimulator")

## 2. Data Preparation

We load and scale the same financial data as in the QAOA notebook.

In [ ]:
full_returns = np.load("2016-01-01_returns.npy")
correlation_matrix = np.load("2016-01-01_correlation.npy")
covariance_matrix = np.load("2016-01-01_covariance.npy")

# Scale values for quantum algorithms (×10,000 brings ~0.0001 values to ~1.0)
scale_factor = 10000
scaled_full_returns = full_returns * scale_factor
scaled_covariance_matrix = covariance_matrix * scale_factor
# Correlation matrix is already normalized to [-1, 1] — don't scale
scaled_correlation_matrix = correlation_matrix

print(f"Portfolio size: {len(full_returns)} assets")

## 3. Partitioning

We partition the assets using spectral clustering on the correlation matrix, same as the QAOA approach.

In [ ]:
from modularity_spectral_partitioning import modularity_spectral_threshold
from visualization import plot_partition_counts

MAX_PARTITION_SIZE = 20

communities, partitions = modularity_spectral_threshold(
    correlation_matrix, threshold=MAX_PARTITION_SIZE
)

plot_partition_counts(correlation_matrix, partitions, threshold=MAX_PARTITION_SIZE)

In [ ]:
# Extract sub-problems for each partition
partitioned_covariance_matrices = {}
partitioned_returns = {}

for cluster_label in np.unique(partitions):
    cluster_indices = np.flatnonzero(partitions == cluster_label)
    partitioned_covariance_matrices[cluster_label] = scaled_covariance_matrix[
        np.ix_(cluster_indices, cluster_indices)
    ]
    partitioned_returns[cluster_label] = scaled_full_returns[cluster_indices]

print(f"Created {len(partitioned_returns)} partitions")
for pid, ret in partitioned_returns.items():
    print(f"  Partition {pid}: {len(ret)} assets")

## 4. Build QUBO Matrices

Same QUBO formulation as the QAOA notebook: **Minimize Risk − λ·Return**.

In [ ]:
from utils import build_qubo_matrices_for_all_partitions

LAMBDA_PARAM = 0.75

qubo_matrices = build_qubo_matrices_for_all_partitions(
    partitioned_returns_dict=partitioned_returns,
    partitioned_covariance_dict=partitioned_covariance_matrices,
    lambda_param=LAMBDA_PARAM,
)

print(f"Built QUBO matrices for {len(qubo_matrices)} partitions")

## 5. Solve with PCE

Now, instead of using QAOA (1 qubit per variable), we use **PCE** to solve each partition's QUBO with dramatically fewer qubits.

PCE works by:
1. Mapping each binary variable to the **parity of a qubit subset** (via Pauli masks)
2. Using a parameterized circuit (UCCSD ansatz) to find low-energy states
3. Decoding the measurement outcomes back to the original variable space

### Single Partition Example

Let's first demonstrate PCE on a single partition to understand the API.

In [ ]:
# Pick a partition to demonstrate
demo_partition_id = list(qubo_matrices.keys())[0]
demo_qubo = qubo_matrices[demo_partition_id]
n_vars = demo_qubo.shape[0]

print(f"Partition {demo_partition_id}: {n_vars} variables (assets)")
print(f"  QAOA would require: {n_vars} qubits")

In [ ]:
# Create PCE instance with dense encoding (logarithmic compression)
pce = PCE(
    qubo_matrix=demo_qubo,
    encoding_type="dense",         # O(log₂ N) qubits
    alpha=2.0,                     # Softness parameter
    n_layers=2,                    # Circuit depth
    n_electrons=2,                 # Initial placeholder
    max_iterations=15,
    backend=backend,               # Uses QoroService or ParallelSimulator
    optimizer=PymooOptimizer(method=PymooMethod.DE, population_size=20),
)

# Set n_electrons properly (must be even and < n_qubits)
n_elec = max(2, (pce.n_qubits - 2) // 2 * 2)
pce.n_electrons = n_elec

print(f"  PCE uses: {pce.n_qubits} qubits ({n_vars / pce.n_qubits:.1f}× compression)")

In [ ]:
# Run PCE optimization
pce.run()

pce_solution = pce.solution
print(f"\nPCE solution: {pce_solution}")
print(f"Assets selected: {sum(pce_solution)} / {n_vars}")

### Evaluate the PCE Solution

In [ ]:
from utils import evaluate_solution

evaluate_solution(
    qubo_matrix=qubo_matrices[demo_partition_id],
    qaoa_solution=pce_solution,
    returns=partitioned_returns[demo_partition_id],
    covariance_matrix=partitioned_covariance_matrices[demo_partition_id],
    partition_id=demo_partition_id,
)

## 6. Solve All Partitions with PCE

Now let's solve all partitions and aggregate into a global portfolio. We solve each partition sequentially using PCE with the selected backend.

In [ ]:
from utils import aggregate_partition_solutions

pce_partition_solutions = {}

for pid, qubo in qubo_matrices.items():
    n_vars = qubo.shape[0]

    pce_inst = PCE(
        qubo_matrix=qubo,
        encoding_type="dense",
        alpha=2.0,
        n_layers=2,
        n_electrons=2,
        max_iterations=15,
        backend=backend,  # Uses QoroService or ParallelSimulator
        optimizer=PymooOptimizer(method=PymooMethod.DE, population_size=20),
    )
    n_elec = max(2, (pce_inst.n_qubits - 2) // 2 * 2)
    pce_inst.n_electrons = n_elec

    print(f"Partition {pid}: {n_vars} variables → {pce_inst.n_qubits} qubits")
    pce_inst.run()
    pce_partition_solutions[pid] = pce_inst.solution

print(f"\nSolved all {len(pce_partition_solutions)} partitions with PCE")

In [ ]:
# Aggregate partition solutions into a global portfolio
pce_bitstring = aggregate_partition_solutions(pce_partition_solutions, partitions)

print(f"Global portfolio: {sum(pce_bitstring)} / {len(pce_bitstring)} assets selected")

## 7. Compare PCE vs Exact Solver

In [ ]:
from utils import solve_and_aggregate_partitions, compare_portfolio_solutions

# Get the classical exact solution for comparison
exact_bitstring = solve_and_aggregate_partitions(qubo_matrices, partitions)

In [ ]:
compare_portfolio_solutions(
    pce_bitstring,
    exact_bitstring,
    scaled_full_returns,
    scaled_covariance_matrix,
)

## 8. Qubit Savings Summary

Let's summarize the qubit savings across all partitions.

In [ ]:
total_qaoa_qubits = sum(q.shape[0] for q in qubo_matrices.values())

# Estimate PCE qubit counts
total_pce_dense_qubits = sum(
    max(2, math.ceil(math.log2(q.shape[0])) + 1)
    for q in qubo_matrices.values()
)

print("Qubit Requirements Comparison")
print("=" * 45)
print(f"{'Method':<20} {'Total Qubits':>12} {'Max per Partition':>18}")
print("-" * 45)
print(f"{'QAOA':<20} {total_qaoa_qubits:>12} {max(q.shape[0] for q in qubo_matrices.values()):>18}")
print(f"{'PCE (dense)':<20} {total_pce_dense_qubits:>12} {max(max(2, math.ceil(math.log2(q.shape[0])) + 1) for q in qubo_matrices.values()):>18}")
print(f"\nCompression ratio: {total_qaoa_qubits / total_pce_dense_qubits:.1f}×")
print(f"\n💡 With QoroService, even 100+ asset portfolios need only ~7 qubits per partition!")

## Summary

**PCE (Pauli Correlation Encoding)** provides a compelling alternative to QAOA for portfolio optimization:

| Aspect | QAOA | PCE (Dense) |
|--------|------|-------------|
| Qubits per partition | N (# assets) | O(log₂ N) |
| Circuit type | QAOA mixer + cost | UCCSD ansatz |
| Solution quality | High | Competitive |
| Hardware ready | Limited by qubit count | Dramatically fewer qubits |

**When to use PCE over QAOA:**
- Large partitions that exceed available qubit counts
- Near-term hardware with limited connectivity
- When qubit savings outweigh potential quality trade-offs

**Cloud execution with QoroService** is the recommended path for production workloads — simply set `USE_CLOUD = True` above.